# RetractorDB in Colab: a stream feeding a PyTorch training loop

J5 of the embedded programme (`docs/jupyter-integration.md`). The engine comes from a
wheel, compiles a continuous query in-process and hands its output windows to PyTorch
through DLPack. No daemon, no build step, no runtime restart.

**Where it runs.** Google Colab, or any Linux x86_64 / aarch64 Jupyter kernel with
Python 3.10 or later - the wheels are Linux-only for now. For a build tree instead of
a wheel, see `j1_engine.ipynb`.

**The first cell is the acceptance test of the packaging:** install plus import in
under about 20 seconds, and no prompt to restart the runtime. `retractordb` depends on
`numpy>=1.23` only, not on torch, so pip leaves Colab's own numpy and torch alone.
`%pip` still ends with IPython's stock *you may need to restart the kernel*; nothing
here needs it, because the only package pip touches is `retractordb` itself.

In [ ]:
import os
import time
from importlib.metadata import version

# retractordb is not on PyPI yet. Until it is, upload the Linux cp312-abi3 wheel with
# the Files panel and name it here (a shell pattern is fine), for example
# WHEEL = '/content/retractordb-*-cp312-abi3-*_x86_64.whl'
WHEEL = os.environ.get('RDB_WHEEL', 'retractordb')

started = time.perf_counter()
%pip install --quiet {WHEEL}
import retractordb as rdb
print(rdb.Engine)  # the first embedded name loads the compiled engine
print(f"retractordb {version('retractordb')}: install + import {time.perf_counter() - started:.1f} s")

## A signal

A stand-in for a sensor: 200 seconds of a 0.5 Hz sine sampled at 50 Hz, with noise.
The plan reads it from a text file, one value per line - the same `FILE` source a
deployment points at a real log.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

workdir = Path(tempfile.mkdtemp(prefix='rdb-'))  # the engine's storage directory

rng = np.random.default_rng(0)
t = np.arange(10_000) / 50  # 200 s at 50 Hz
signal = np.sin(2 * np.pi * 0.5 * t) + 0.3 * rng.standard_normal(t.size)
(workdir / 'sensor.txt').write_text(''.join(f'{v:.6f}\n' for v in signal))
print(workdir, signal.size, 'samples')

## The plan

`raw` samples the file every 1/50 s; `smoothed` is its moving average over the last
100 records. `compile()` parses and compiles the plan and runs nothing. `run(slots=n)`
advances `n` time slots with the GIL released - the stop button interrupts it.

In [ ]:
eng = rdb.Engine(str(workdir))
eng.compile(f"""
DECLARE v FLOAT STREAM raw, 1/50 FILE '{workdir / 'sensor.txt'}'
SELECT AVG(v:100) STREAM smoothed FROM raw
""")
print(eng.streams(), eng.schema('smoothed'))

eng.run(slots=5_000)  # the first half of the input
print('slots:', eng.slots_done, ' plan time:', eng.time, ' smoothed records:', eng.record_count('smoothed'))

## Windows into tensors

`window()` stacks windows of `size` records `stride` apart: shape
`(n_windows, size, n_values)`. That is one copy out of the engine. `torch.from_dlpack`
shares the copy instead of making another, so the tensor stays valid while the engine
keeps running.

Colab ships torch. Anywhere else, install it yourself - `retractordb` never imports it
for you.

In [ ]:
import torch

w = eng.window('smoothed', fields=['smoothed_0'], size=100, stride=1, dtype='float32')
x = torch.from_dlpack(w)
print(w)
print(x.shape, x.dtype, '- shares memory with the window:', x.data_ptr() == w.to_numpy().ctypes.data)

# The engine's AVG against numpy's moving average over the same input
source = np.loadtxt(workdir / 'sensor.txt')
expected = np.convolve(source, np.ones(100) / 100, mode='valid')[: eng.record_count('smoothed')]
assert np.allclose(eng.to_numpy('smoothed')[:, 0], expected, atol=1e-5)

## A training loop fed by the engine

`StreamDataset` is an `IterableDataset` over the same windows, re-read at every epoch.
Each epoch below first advances the engine by 1,000 slots, so every epoch trains on a
longer history - the engine as a data source that keeps producing. The model is
deliberately tiny: a linear map from 100 smoothed values to the next one.

`num_workers=0` is structural, not a placeholder: DataLoader workers are forked
processes, and the engine does not survive `fork`.

In [ ]:
from torch.utils.data import DataLoader

from retractordb.torch import StreamDataset

ds = StreamDataset(eng, 'smoothed', fields=['smoothed_0'], window=101, stride=1)
loader = DataLoader(ds, batch_size=256, num_workers=0)

torch.manual_seed(0)
model = torch.nn.Linear(100, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

epoch = 0
while not eng.end_of_input:
    eng.run(slots=1_000)
    total, batches = 0.0, 0
    for batch in loader:  # (batch, 101, 1)
        inputs, target = batch[:, :-1, 0], batch[:, -1:, 0]
        loss = torch.nn.functional.mse_loss(model(inputs), target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total, batches = total + loss.item(), batches + 1
    print(f'epoch {epoch}: {len(ds):5} windows, mse {total / batches:.6f}')
    epoch += 1

## Done

`close()` releases the plan's storages; `with rdb.Engine(...) as eng:` does the same on
exit. `j1_engine.ipynb` covers the rest of the API - `step()`, records, `to_numpy()`,
and what raises.

In [ ]:
eng.close()